# 70 - ablations on the T4 (ARCHITECTURE section 10.8)
Turn one feature off (or on) at a time and measure batch-1 tokens per second and the 32-client run (throughput, TTFT p90, TPOT p90). Configurations: **baseline**, **no CUDA graphs**, **no chunked prefill** (`--max-num-batched-tokens 4096`), **no prefix caching**, **INT8 weights**, **n-gram speculation**, **Autopilot off** (no startup kernel tuning) and **reference kernels** (torch ops instead of Triton).

Short runs on purpose (24 prompts at batch 1, 192 prompts at 32 clients, one repetition; the reference-kernel run is smaller still), so the whole notebook takes about an hour. Run on a **GPU T4 x2** session with Internet on and Persistence = Variables and Files, then press **Run All** (or Save Version for an unattended batch run). A configuration that fails or exceeds its time limit is recorded and skipped. Do not use *Factory reset*.

In [ ]:
import os, sys, json, time, signal, shutil, subprocess, threading, glob

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working"
MODEL = f"{WORK}/models/Qwen2.5-1.5B-Instruct"
SHAREGPT = f"{WORK}/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
REPO = f"{WORK}/turingserve"
RESULTS = f"{WORK}/results"
PORT = 8000
os.makedirs(RESULTS, exist_ok=True)
os.environ["TS_SHAREGPT"] = SHAREGPT
os.environ["TS_KAGGLE"] = "1"

# name, extra `turingserve serve` flags, prompts for the batch-1 run, prompts for the 32-client run, time limit per benchmark (s),
# time allowed for the server to start (s)
# Short runs on purpose: enough to see what each feature buys, small enough to finish in about an hour.
CONFIGS = [
    ("baseline",           [],                                                         24, 192, 1200, 1200),
    ("no-cuda-graphs",     ["--no-cuda-graphs"],                                       24, 192, 1200, 1200),
    ("no-chunked-prefill", ["--max-num-batched-tokens", "4096"],                       24, 192, 1200, 1200),
    ("no-prefix-caching",  ["--no-prefix-caching"],                                    24, 192, 1200, 1200),
    ("int8",               ["--quantization", "int8"],                                 24, 192, 1200, 1200),
    ("ngram-speculation",  ["--spec-method", "ngram", "--spec-k", "4"],                24, 192, 1200, 1200),
    ("autopilot-off",      ["--autopilot", "off"],                                     24, 192, 1200, 1200),
    ("reference-kernels",  ["--kernels", "reference"],                                  4,  32, 900, 420),   # torch ops, a debug path: may not start in time
]
srv = None


def gpu_used_mib():
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                       capture_output=True, text=True)
    return int(r.stdout.strip().splitlines()[0]) if r.returncode == 0 and r.stdout.strip() else -1


def stop_server():
    global srv
    if srv is not None and srv.poll() is None:
        try:
            os.killpg(os.getpgid(srv.pid), signal.SIGTERM)
            srv.wait(timeout=30)
        except Exception:
            try:
                os.killpg(os.getpgid(srv.pid), signal.SIGKILL)
            except Exception:
                pass
    srv = None
    subprocess.run(["pkill", "-f", "turingserve serve"], capture_output=True)
    time.sleep(3)
    for _ in range(30):
        if 0 <= gpu_used_mib() < 500:
            break
        time.sleep(2)


def start_server(name, extra, wait=1200):
    """Launch TuringServe with the given flags and wait for /health (the first start also tunes the kernels)."""
    global srv
    import requests
    stop_server()
    log_path = f"{WORK}/server_{name}.log"
    srv = subprocess.Popen(["turingserve", "serve", "--model", MODEL, "--served-model-name", "qwen", "--port", str(PORT),
                            "--max-model-len", "4096", *extra], stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                           start_new_session=True)
    t0 = time.time()
    for i in range(max(1, wait // 5)):
        try:
            if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
                print(f"[{name}] server up after {time.time() - t0:.0f}s")
                return True, ""
        except Exception:
            pass
        if srv.poll() is not None:
            break
        if i % 12 == 11:
            print(f"  [{name}] still starting... {time.time() - t0:.0f}s", flush=True)
        time.sleep(5)
    txt = open(log_path, errors="replace").read()
    print(f"[{name}] did not start; log tail:\n{txt[-1500:]}")
    hits = [l for l in txt.splitlines() if "rror" in l]
    stop_server()
    return False, (hits[-1] if hits else f"not up after {wait} s; last log line: {txt.strip().splitlines()[-1] if txt.strip() else ''}")[:240]


def run_bench(label, extra, limit):
    """serve_bench with a time limit; returns True when it finished."""
    cmd = [sys.executable, "-u", "-m", "benchmarks.serve_bench", "--base-url", f"http://localhost:{PORT}", "--engine", "turingserve",
           "--model", "qwen", "--tokenizer", MODEL, "--dataset", "sharegpt", "--mode", "concurrency", "--repeat", "1", "--warmup", "4",
           "--ignore-eos", "--out", RESULTS, "--label", label, *extra]
    p = subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    timer = threading.Timer(limit, p.kill)
    timer.start()
    for line in p.stdout:
        if "Token indices sequence length" in line or "Running this sequence" in line:
            continue
        print(line, end="", flush=True)
    p.wait()
    timer.cancel()
    ok = p.returncode == 0
    print(f"[{label}] {'finished' if ok else f'stopped (exit {p.returncode}, limit {limit}s)'}\n")
    return ok


def result_exists(label):
    return bool(glob.glob(f"{RESULTS}/*_turingserve_sharegpt_{label}.json"))


print("setup ok; GPU 0 memory used:", gpu_used_mib(), "MiB;", len(CONFIGS), "configurations")

## 1. Pin the stack (torch 2.6.0+cu124, Triton 3.2.0)

In [ ]:
def torch_version():
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True)
    return r.stdout.strip()

print("torch before:", torch_version())
if torch_version() != "2.6.0+cu124":
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], capture_output=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url",
                        "https://download.pytorch.org/whl/cu124"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError("torch pin failed")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45,<5", "safetensors", "fastapi",
                    "uvicorn[standard]", "httpx", "openai", "prometheus-client", "psutil", "datasets", "pandas", "matplotlib"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])
    raise RuntimeError("dependency install failed")
print("torch after:", torch_version())

## 2. Model, dataset and repository at tag `m3-rc`

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

if not os.path.exists(f"{MODEL}/model.safetensors"):
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL)
if not os.path.exists(SHAREGPT):
    hf_hub_download("anon8231489123/ShareGPT_Vicuna_unfiltered", "ShareGPT_V3_unfiltered_cleaned_split.json",
                    repo_type="dataset", local_dir=f"{WORK}/datasets/sharegpt")
if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "https://github.com/Muskanpandey0304/turingserve.git", REPO], check=True)
subprocess.run(["git", "fetch", "--tags", "--force"], cwd=REPO, check=True, capture_output=True)
subprocess.run(["git", "checkout", "-f", "m3-rc"], cwd=REPO, check=True, capture_output=True)
print("repo at:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO, capture_output=True)
r = subprocess.run([sys.executable, "scripts/check_versions.py"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-600:])
open(f"{RESULTS}/versions.txt", "w").write(r.stdout)
print("model ok:", os.path.exists(f"{MODEL}/model.safetensors"), "| dataset ok:", os.path.exists(SHAREGPT))

## 3. Run every configuration
The first server start also tunes the kernels (several minutes); later starts reuse the cache.

In [ ]:
outcomes = []
for name, extra, n1, n32, limit, wait in CONFIGS:
    print("=" * 70, f"\nconfiguration: {name}   flags: {extra or 'defaults'}")
    row = {"name": name, "flags": extra, "started": False, "c1": False, "c32": False, "note": ""}
    if result_exists(f"ablation-{name}-c1") and result_exists(f"ablation-{name}-c32"):
        print("results already exist, skipping")
        row.update(started=True, c1=True, c32=True, note="already measured")
        outcomes.append(row)
        continue
    ok, err = start_server(name, extra, wait)
    row["started"] = ok
    if not ok:
        row["note"] = f"server did not start: {err}"
        outcomes.append(row)
        continue
    row["c1"] = run_bench(f"ablation-{name}-c1", ["--num-prompts", str(n1), "--concurrency", "1"], limit)
    row["c32"] = run_bench(f"ablation-{name}-c32", ["--num-prompts", str(n32), "--concurrency", "32"], limit)
    outcomes.append(row)
    stop_server()
json.dump(outcomes, open(f"{RESULTS}/ablation_outcomes.json", "w"), indent=1)
for o in outcomes:
    print(o)

## 4. Summary and download
Download **`/kaggle/working/ablation_bundle.zip`**, then stop the session.

In [ ]:
stop_server()


def stats(name, level):
    f = glob.glob(f"{RESULTS}/*_turingserve_sharegpt_ablation-{name}-{level}.json")
    if not f:
        return None
    r = json.load(open(f[0]))["runs"][0]["median"]
    return r


b1, b32 = stats("baseline", "c1"), stats("baseline", "c32")
print(f"{'configuration':20}{'batch-1 tok/s':>14}{'32 clients tok/s':>18}{'vs baseline':>13}{'TTFT p90 ms':>13}{'TPOT p90 ms':>13}{'errors':>8}")
for name, *_ in CONFIGS:
    c1, c32 = stats(name, "c1"), stats(name, "c32")
    if c1 is None and c32 is None:
        print(f"{name:20}  not measured")
        continue
    t1 = f"{c1['output_throughput_tok_s']:.1f}" if c1 else "-"
    t32 = f"{c32['output_throughput_tok_s']:.1f}" if c32 else "-"
    rel = f"{c32['output_throughput_tok_s'] / b32['output_throughput_tok_s']:.2f}x" if (c32 and b32) else "-"
    ttft = f"{c32['ttft']['p90'] * 1000:.0f}" if c32 else "-"
    tpot = f"{c32['tpot']['p90'] * 1000:.1f}" if c32 else "-"
    err = (c1["errors"] if c1 else 0) + (c32["errors"] if c32 else 0)
    print(f"{name:20}{t1:>14}{t32:>18}{rel:>13}{ttft:>13}{tpot:>13}{err:>8}")

bundle = f"{WORK}/ablation_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in glob.glob(f"{RESULTS}/*ablation*") + glob.glob(f"{RESULTS}/versions.txt") + glob.glob(f"{WORK}/server_*.log"):
    if os.path.isfile(f):
        shutil.copy(f, bundle)
shutil.make_archive(bundle, "zip", bundle)
print("\nDownload:", bundle + ".zip", f"({os.path.getsize(bundle + '.zip') / 1e6:.1f} MB)")